In [ ]:
import random
from pathlib import Path
from main import load_dotenv
from data_loader.raw_dataloader import RawDataloader
from models.ngram.knn import KNN
from models.ngram.model import Model
from collections import Counter, defaultdict

load_dotenv(Path("../.env"))

In [ ]:
dataloader = RawDataloader()
dataloader.prepare_dataset(representation="remi")
loader = dataloader.loader(load_music=dataloader.load_music)

In [ ]:
ngram_model = Model(loader=loader, n=5)
ngram_model.remi_train()

In [ ]:
ngram_model.models[0]

In [ ]:
next_token = None
tokens = (-1, -1,-1, -1)

while next_token != -1000:
    next_token = ngram_model.remi_predict(tokens)[0]
    tokens += (next_token,)

In [ ]:
response = tokens[4:-1]

In [ ]:
from data_processing.music_representations.decoders import DecodeContext, create_decoder
from data_processing.music_representations.helpers.adapters import canonical_frames_to_midi

# Generated tuples -> canonical -> MIDI. Nothing goes straight to MIDI: canonical is
# the only thing that writes a .mid, so this is the same path the built dataset takes.
decoder = create_decoder("remi", dataloader.config, dataloader.storage)
frames = decoder.decode(response, DecodeContext(piece_id="ngram_sample"))
frames["notes"].head()

In [ ]:
midi_path = canonical_frames_to_midi(frames, Path("../generated/remi_sample.mid"))
print(midi_path, frames["notes"].height, "notes")
